# 04. Iterative AI Answer Improvement Workflow

## Overview
Welcome to the **Iterative AI Answer Improvement Workflow** project! In previous projects, you explored sequential workflows where each node executes exactly once in a linear chain:

$$\text{START} \longrightarrow \text{Node A} \longrightarrow \text{Node B} \longrightarrow \text{Node C} \longrightarrow \text{END}$$

In real-world AI applications, a single-pass response often misses key technical details, contains ambiguities, or lacks structure. This project introduces **iterative workflows with loops and reflection patterns** in **LangGraph**.

Here, the system evaluates its own output, identifies weaknesses, refines the answer, and repeats the evaluation until a target quality threshold is achieved or a maximum retry limit is reached.

---
## 🔄 Architecture & Flow

```
                              START
                                │
                                ▼
                         Generate Answer
                                │
                                ▼
                           Critic Node
                                │
                                ▼
                         Quality Evaluation
                                │
                    ┌───────────┴───────────┐
                    │                       │
                    ▼                       ▼
               Good Enough             Needs Improvement
              (Score >= 8)               (Score < 8)
                    │                       │
                    ▼                       ▼
                   END                Improve Answer
                                           │
                                           │
                                           └───────────────┐
                                                           │
                                                           ▼
                                                     Critic Again
```

### Core Concepts Introduced:
- **Reflection Pattern**: Separating generation and critique into dedicated specialist nodes.
- **Cyclic Graphs**: Routing back from `improve_answer` to `critic`.
- **Structured Output Evaluation**: Using Pydantic to score responses (1–10) and extract actionable feedback.
- **Loop Termination**: Dual safeguard (`Score >= target_score` OR `Iteration >= max_iterations`) to prevent infinite loops.

--- 
## 1. Setup & Dependencies

Let's install and import the required libraries: `langgraph`, `langchain-google-genai`, `pydantic`, `python-dotenv`, and `rich`.

In [ ]:
import os
import getpass
from typing import TypedDict, List, Dict, Any, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# LangChain & LangGraph
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display, Markdown

# Load environment variables
load_dotenv()

# Check for Google Gemini API Key
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not api_key or api_key == "your_google_api_key_here":
    api_key = getpass.getpass("Enter your Google Gemini API Key: ")
    os.environ["GOOGLE_API_KEY"] = api_key

print("✅ Environment initialized successfully!")

--- 
## 2. LLM Initialization

We use Google's **Gemini 2.5 Flash** (or configurable via `GEMINI_MODEL`), which delivers fast response times, strong reasoning, and native support for structured schema outputs.

In [ ]:
model_name = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# Standard generator LLM (higher temperature for creative drafting)
generator_llm = ChatGoogleGenerativeAI(
    model=model_name,
    temperature=0.7,
    google_api_key=os.environ.get("GOOGLE_API_KEY")
)

# Evaluator / Critic LLM (lower temperature for objective, deterministic grading)
critic_llm = ChatGoogleGenerativeAI(
    model=model_name,
    temperature=0.2,
    google_api_key=os.environ.get("GOOGLE_API_KEY")
)

print(f"🤖 Connected to model: {model_name}")

--- 
## 3. State Management & Structured Schema

### State Structure (`WorkflowState`)
The shared state carries the entire lifecycle of the conversation:
- `question`: The original query.
- `answer`: The current working version of the response.
- `score`: Quality rating from 1 to 10 assigned by the critic.
- `critique`: Specific actionable instructions for improvement.
- `strengths`: Notable positive aspects of the answer.
- `weaknesses`: Specific gaps, missing points, or inaccuracies.
- `iteration`: The count of completed improvement cycles.
- `max_iterations`: Hard upper bound to prevent infinite loops.
- `target_score`: Threshold required to accept the answer.
- `history`: Full audit trail of iterations and score progression.

In [ ]:
class CritiqueOutput(BaseModel):
    """Structured grading schema for the Critic Node."""
    score: int = Field(
        description="Overall quality score from 1 to 10 evaluating correctness, clarity, depth, and completeness.",
        ge=1,
        le=10
    )
    strengths: List[str] = Field(
        description="Key positive aspects and accurate elements of the answer."
    )
    weaknesses: List[str] = Field(
        description="Specific missing details, ambiguities, inaccuracies, or structural issues."
    )
    critique: str = Field(
        description="Concrete, actionable recommendations on how to refine the answer to achieve a 10/10."
    )

class WorkflowState(TypedDict):
    """Shared state dictionary passed between LangGraph nodes."""
    question: str
    answer: str
    critique: str
    score: int
    strengths: List[str]
    weaknesses: List[str]
    iteration: int
    max_iterations: int
    target_score: int
    history: List[Dict[str, Any]]

print("📋 Schemas defined successfully!")

--- 
## 4. Prompt Engineering

We define three specialized prompt templates:
1. **`GENERATE_PROMPT`**: Guides initial drafting.
2. **`CRITIC_PROMPT`**: Establishes strict evaluation rubrics (Accuracy, Clarity, Depth, Relevance).
3. **`IMPROVE_PROMPT`**: Instructs the model to synthesize the previous draft + weaknesses + critic advice.

In [ ]:
GENERATE_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an AI assistant. Provide a direct, concise initial answer to the user's question. "
        "Provide a solid baseline draft."
    ),
    ("user", "Question: {question}")
])

CRITIC_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an objective, rigorous, and constructive expert reviewer.\n"
        "Your task is to evaluate the provided answer to the user's question.\n\n"
        "Evaluation Criteria:\n"
        "1. Accuracy & Correctness: Is the explanation factually accurate and technically sound?\n"
        "2. Clarity & Structure: Is it well-organized, readable, and easy to follow?\n"
        "3. Completeness & Depth: Does it explain key mechanisms, underlying reasons, and practical value?\n"
        "4. Relevance: Does it directly address the specific question?\n\n"
        "Scoring Scale (1-10):\n"
        "- 1-4: Poor, superficial, incomplete, or contains inaccuracies.\n"
        "- 5-7: Decent/Average, answers the question but lacks technical depth or thorough explanation.\n"
        "- 8-10: Excellent, comprehensive, well-structured, clear, and informative.\n\n"
        "Target Quality Threshold: {target_score}/10."
    ),
    (
        "user",
        "User Question:\n{question}\n\n"
        "Current Answer to Evaluate:\n{answer}\n\n"
        "Current Iteration: {iteration}\n"
        "Please provide your structured evaluation."
    )
])

IMPROVE_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert AI editor and response refiner.\n"
        "Your goal is to significantly improve the previous answer by acting on the critic's feedback.\n\n"
        "Guidelines for improvement:\n"
        "1. Address every weakness identified by the critic.\n"
        "2. Preserve and build upon the accurate strengths of the previous answer.\n"
        "3. Enhance technical precision, structure (use clear headings/bullet points where appropriate), and depth.\n"
        "4. Output only the revised, improved answer."
    ),
    (
        "user",
        "Original Question:\n{question}\n\n"
        "Previous Answer (Score: {score}/10):\n{answer}\n\n"
        "Critic Strengths:\n{strengths}\n\n"
        "Critic Weaknesses:\n{weaknesses}\n\n"
        "Critic Actionable Feedback:\n{critique}\n\n"
        "Generate the improved version of the answer now:"
    )
])

print("📝 Prompt templates ready!")

--- 
## 5. Node Functions Implementation

Now we implement the three graph nodes:
1. **`generate_initial_answer`**: Creates the baseline draft and initializes history.
2. **`critic_evaluation`**: Employs `critic_llm.with_structured_output(CritiqueOutput)` for schema validation.
3. **`improve_answer`**: Synthesizes the improvement prompt and increments the `iteration` counter.

In [ ]:
def generate_initial_answer(state: WorkflowState) -> Dict[str, Any]:
    """Node 1: Generates initial answer draft."""
    print(f"\n[Node 1: Generate Answer] Generating initial draft for question...")
    chain = GENERATE_PROMPT | generator_llm
    response = chain.invoke({"question": state["question"]})
    answer_text = response.content.strip()
    
    history_entry = {
        "step": "Initial Generation",
        "iteration": 0,
        "answer": answer_text,
        "score": None,
        "critique": None
    }
    
    return {
        "answer": answer_text,
        "iteration": 0,
        "history": [history_entry]
    }

def critic_evaluation(state: WorkflowState) -> Dict[str, Any]:
    """Node 2: Evaluates the current answer against quality criteria."""
    iteration = state.get("iteration", 0)
    target_score = state.get("target_score", 8)
    print(f"\n[Node 2: Critic Evaluation] Assessing answer (Iteration {iteration})...")
    
    structured_critic = critic_llm.with_structured_output(CritiqueOutput)
    chain = CRITIC_PROMPT | structured_critic
    
    try:
        feedback: CritiqueOutput = chain.invoke({
            "question": state["question"],
            "answer": state["answer"],
            "iteration": iteration,
            "target_score": target_score
        })
        score = feedback.score
        critique = feedback.critique
        strengths = feedback.strengths
        weaknesses = feedback.weaknesses
    except Exception as e:
        print(f"⚠️ Structured output fallback: {e}")
        score = 6
        critique = "Ensure more depth, structured sections, and clear reasoning."
        strengths = ["Answers prompt directly."]
        weaknesses = ["Needs deeper technical explanation."]
        
    print(f"   ⭐ Quality Score: {score}/10 (Target: {target_score})")
    print(f"   💡 Critique: {critique[:120]}...")
    
    history_entry = {
        "step": f"Critic Evaluation (Iteration {iteration})",
        "iteration": iteration,
        "answer": state["answer"],
        "score": score,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "critique": critique
    }
    
    current_history = list(state.get("history", []))
    current_history.append(history_entry)
    
    return {
        "score": score,
        "critique": critique,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "history": current_history
    }

def improve_answer(state: WorkflowState) -> Dict[str, Any]:
    """Node 3: Refines the answer using critique feedback and increments iteration."""
    next_iteration = state["iteration"] + 1
    print(f"\n[Node 3: Improve Answer] Refining response for Iteration {next_iteration}...")
    
    chain = IMPROVE_PROMPT | generator_llm
    strengths_str = "\n- ".join(state.get("strengths", [])) if state.get("strengths") else "None."
    weaknesses_str = "\n- ".join(state.get("weaknesses", [])) if state.get("weaknesses") else "None."
    
    response = chain.invoke({
        "question": state["question"],
        "answer": state["answer"],
        "score": state.get("score", 0),
        "strengths": f"- {strengths_str}",
        "weaknesses": f"- {weaknesses_str}",
        "critique": state.get("critique", "")
    })
    
    improved_text = response.content.strip()
    
    history_entry = {
        "step": f"Answer Improvement (Iteration {next_iteration})",
        "iteration": next_iteration,
        "answer": improved_text,
        "score": None,
        "critique": None
    }
    
    current_history = list(state.get("history", []))
    current_history.append(history_entry)
    
    return {
        "answer": improved_text,
        "iteration": next_iteration,
        "history": current_history
    }

print("⚙️ Nodes defined successfully!")

--- 
## 6. Conditional Routing & Termination Logic

To prevent infinite loops, we define a routing function that checks:
1. **Quality threshold reached** (`score >= target_score`): Response is high quality $\rightarrow$ Route to `END`.
2. **Safety iteration limit reached** (`iteration >= max_iterations`): Maximum retries exhausted $\rightarrow$ Route to `END`.
3. **Needs improvement**: Route to `improve_answer`.

In [ ]:
def route_decision(state: WorkflowState) -> str:
    """Determines whether to finish or iterate."""
    score = state.get("score", 0)
    iteration = state.get("iteration", 0)
    max_iterations = state.get("max_iterations", 3)
    target_score = state.get("target_score", 8)
    
    if score >= target_score:
        print(f"\n🎯 [Routing Decision] Target score reached ({score}/{target_score})! Routing to END.")
        return "good_enough"
    elif iteration >= max_iterations:
        print(f"\n🛑 [Routing Decision] Max iterations reached ({iteration}/{max_iterations}). Routing to END.")
        return "max_reached"
    else:
        print(f"\n🔁 [Routing Decision] Score ({score}/{target_score}) below target. Routing to improve_answer.")
        return "needs_improvement"

print("🔀 Router function defined!")

--- 
## 7. Graph Assembly & Compilation

We now assemble our LangGraph `StateGraph`, register nodes, wire standard and conditional edges, and compile the workflow.

In [ ]:
workflow = StateGraph(WorkflowState)

# 1. Add Nodes
workflow.add_node("generate_answer", generate_initial_answer)
workflow.add_node("critic", critic_evaluation)
workflow.add_node("improve_answer", improve_answer)

# 2. Wire Initial Flow
workflow.add_edge(START, "generate_answer")
workflow.add_edge("generate_answer", "critic")

# 3. Add Conditional Routing
workflow.add_conditional_edges(
    "critic",
    route_decision,
    {
        "good_enough": END,
        "max_reached": END,
        "needs_improvement": "improve_answer"
    }
)

# 4. Wire Loop: improve_answer -> critic
workflow.add_edge("improve_answer", "critic")

# 5. Compile
app = workflow.compile()
print("🎉 LangGraph compiled successfully!")

### Visualizing the Compiled Graph
Let's render the computational graph diagram.

In [ ]:
try:
    # Try rendering Mermaid PNG
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Mermaid PNG rendering not available (requires pygraphviz/mermaid API). ASCII diagram:")
    print(app.get_graph().draw_ascii())

--- 
## 8. Executing the Workflow: Test Case 1

Let's test the workflow with a technical question:
> **Question**: *"Explain why Retrieval-Augmented Generation (RAG) is useful in AI applications."*

In [ ]:
initial_state = {
    "question": "Explain why Retrieval-Augmented Generation (RAG) is useful in AI applications.",
    "answer": "",
    "critique": "",
    "score": 0,
    "strengths": [],
    "weaknesses": [],
    "iteration": 0,
    "max_iterations": 3,
    "target_score": 8,
    "history": []
}

print("=" * 75)
print("🚀 Starting Iterative Workflow Execution (Example 1)...")
print("=" * 75)

result_state = None
for output in app.stream(initial_state):
    for node_name, state_update in output.items():
        if result_state is None:
            result_state = dict(initial_state)
        result_state.update(state_update)

print("\n" + "=" * 75)
print(f"✅ Workflow Finished! Final Score: {result_state['score']}/10 across {result_state['iteration']} iteration(s).")
print("=" * 75)

### 📊 Score Trajectory & Iteration History
Let's inspect the step-by-step progress and critique logs across iterations.

In [ ]:
print("\n--- Iteration History ---\n")
for i, item in enumerate(result_state.get("history", [])):
    step_name = item["step"]
    score_disp = f"{item['score']}/10" if item.get("score") is not None else "N/A"
    print(f"Step {i+1}: {step_name} | Score: {score_disp}")
    if item.get("critique"):
        print(f"   Critique: {item['critique']}")
        if item.get("weaknesses"):
            print(f"   Identified Weaknesses: {', '.join(item['weaknesses'])}")
    print("-" * 70)

### 🔍 Side-by-Side Comparison: Initial Draft vs Final Improved Answer

In [ ]:
initial_draft = result_state["history"][0]["answer"]
final_answer = result_state["answer"]

display(Markdown(f"### 📝 Initial Baseline Draft (Iteration 0)\n\n{initial_draft}"))
display(Markdown("---"))
display(Markdown(f"### 🏆 Final Polished Answer (Score: {result_state['score']}/10)\n\n{final_answer}"))

--- 
## 9. Test Case 2: Complex Scenario

Let's run a second scenario on a nuanced, complex topic:  
> **Question**: *"Compare Vector Databases vs Traditional Relational Databases for semantic search and explain when to use which."*

In [ ]:
state_test_2 = {
    "question": "Compare Vector Databases vs Traditional Relational Databases for semantic search and explain when to use which.",
    "answer": "",
    "critique": "",
    "score": 0,
    "strengths": [],
    "weaknesses": [],
    "iteration": 0,
    "max_iterations": 3,
    "target_score": 9,  # High quality bar
    "history": []
}

result_2 = None
for output in app.stream(state_test_2):
    for node_name, state_update in output.items():
        if result_2 is None:
            result_2 = dict(state_test_2)
        result_2.update(state_update)

display(Markdown(f"### 🏆 Final Polished Output (Score: {result_2['score']}/10)\n\n{result_2['answer']}"))

--- 
## 10. Key Takeaways & Real-World Patterns

### Why Iterative Reflection Outperforms Single-Pass Generation
1. **Decoupled Roles**: The generating agent focuses on synthesis, while the critic agent focuses strictly on evaluation, removing bias.
2. **Deterministic Stopping Rules**: Combining quality thresholds with max retry counters prevents runaway token costs and infinite loops.
3. **Extensibility**: This loop pattern forms the foundation of:
   - **Self-Correcting RAG**: Evaluates document relevance $\rightarrow$ rewrites queries if irrelevant.
   - **Code Generation with Test Feedback**: Generates code $\rightarrow$ runs unit tests $\rightarrow$ fixes syntax/logic errors $\rightarrow$ re-tests.
   - **Autonomous Research Agents**: Identifies information gaps $\rightarrow$ performs targeted sub-searches $\rightarrow$ compiles findings.